## Introduction

Notebook for calculating the value of the R* metric as defined in https://mc-stan.org/posterior/reference/rstar.html

In [1]:
from pathlib import Path
import gdown
from MaCh3PythonUtils.diagnostics.rstar import RStar
from MaCh3PythonUtils.file_handling.chain_handler import MultiChainHandler
from rich.console import Console
import numpy as np

Using mps device

In [2]:
# Set up the console for rich output
console = Console()
console.is_jupyter = False # Disable Jupyter mode for rich output

In [3]:
# Download file from google drive
file_url="https://drive.google.com/file/d/1iE6xFhn3BH_HnLUfQ7KFGy2wfeH52Rwf/view?usp=sharing"

# download the file
input_file = Path("../models/demo_chain.root")

if not input_file.exists():
    # download the file
    input_file.parent.mkdir(parents=True, exist_ok=True)
    gdown.download(file_url, str(input_file), quiet=False, fuzzy=True)

In [4]:
# Setup files
file_1 = "../models/demo_chain.root"
file_2 = "../models/demo_chain.root"


chain_handler = MultiChainHandler([file_1, file_2], "posteriors", False, True)
chain_handler.ignore_plots(["LogL_systematic_xsec_cov", "Log", "LogL_systematic_osc_cov"])
chain_handler.add_additional_plots(["sin2th", "delm2", "delta", "xsec"])
chain_handler.add_new_cuts(["LogL_systematic_xsec_cov<1234", "step>10000"])
chain_handler.convert_ttree_to_array()

Attempting to open ../models/demo_chain.root
Succesfully opened ../models/demo_chain.root:posteriors
Attempting to open ../models/demo_chain.root
Succesfully opened ../models/demo_chain.root:posteriors


In [5]:
# Arguments for histboost classifier
MODEL_KWARGS = {
    "learning_rate": 0.01,
    "max_iter": 1000,
    "early_stopping": True,
    "l2_regularization": 0.98,
    "n_iter_no_change": 50,
    "verbose": 0,
    "validation_fraction": 0.2,
    "max_bins": 100
}

In [ ]:
# Do RStar for two identical files

rstar = RStar(chain_handler, n_itejrations=400, **MODEL_KWARGS)
rstar_vals = rstar.get_rstar()
rstar.make_rstar_hist(rstar_vals['train_rstar'], "train_rstar_histogram.pdf")
rstar.make_rstar_hist(rstar_vals['test_rstar'], "test_rstar_histogram.pdf")


Creating 400 models using histboostclassifier algorithm

Creating Models:   0%|          | 0/400 [00:00<?, ?it/s]

Creating 400 different train/test splits...

Assigning data to 400 models in parallel...

Assigning Data:   0%|          | 0/400 [00:00<?, ?it/s]

Using 2 files for R* diagnostics, with a training set containing 39836 entries and a testing set containing 159346 
entries

Initialised RStar with 400 models using histboostclassifier algorithm

Models not trained! Training models first.

Training models of type
HistGradientBoostingClassifier(early_stopping=True, l2_regularization=0.98,
                               learning_rate=0.01, max_bins=100, max_iter=1000,
                               n_iter_no_change=50, validation_fraction=0.2)

Training Models:   0%|          | 0/400 [00:00<?, ?it/s]

In [ ]:
# Setup files
NOISE = 0.1


chain_handler_diff = MultiChainHandler([file_1, file_2], "posteriors", False, True)
chain_handler_diff.ignore_plots(["LogL_systematic_xsec_cov", "Log", "LogL_systematic_osc_cov"])
chain_handler_diff.add_additional_plots(["sin2th", "delm2", "delta", "xsec"])
chain_handler_diff.add_new_cuts(["LogL_systematic_xsec_cov<1234", "step>10000"])
chain_handler_diff.convert_ttree_to_array()

# Now we want to modify the second file slightly
file_2_view = chain_handler_diff.ttree_array[chain_handler_diff.ttree_array["chain_id"]==1]
means = np.mean(file_2_view, axis=0)
random_noise = np.random.uniform(-NOISE, NOISE, size=file_2_view.shape)
# Add random noise to the second file
chain_handler_diff.ttree_array[chain_handler_diff.ttree_array["chain_id"]==1] += random_noise*means

In [ ]:
# Do RStar for noisy data
rstar = RStar(chain_handler_diff, n_iterations=400, **MODEL_KWARGS)
rstar_vals = rstar.get_rstar()
rstar.make_rstar_hist(rstar_vals['train_rstar'], "train_rstar_histogram.pdf")
rstar.make_rstar_hist(rstar_vals['test_rstar'], "test_rstar_histogram.pdf")
